# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
import sys
sys.path.append('../../05_src/')

In [3]:
from openai import OpenAI
import os

#client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
#                    api_key='any value',
#                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

client = OpenAI(api_key=os.getenv("API_GATEWAY_KEY"))

In [4]:
import requests
import os

# Test the gateway directly
url = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/chat/completions"

headers = {
    "Content-Type": "application/json",
    "x-api-key": os.getenv("API_GATEWAY_KEY")
}

payload = {
    "model": "gpt-4o-mini",
    "messages": [{"role": "user", "content": "Say hello"}]
}

response = requests.post(url, headers=headers, json=payload)
print("Status code:", response.status_code)
print("Response:", response.text)

Status code: 200
Response: {
  "id": "chatcmpl-DZgt74gxIBnb3FkiWbYLXwZApxBbO",
  "object": "chat.completion",
  "created": 1777398713,
  "model": "gpt-4o-mini-2024-07-18",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Hello! How can I assist you today?",
        "refusal": null,
        "annotations": []
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 9,
    "completion_tokens": 9,
    "total_tokens": 18,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,
      "audio_tokens": 0,
      "accepted_prediction_tokens": 0,
      "rejected_prediction_tokens": 0
    }
  },
  "service_tier": "default",
  "system_fingerprint": "fp_dd6ab8cf0b"
}



## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF from the URL
pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(pdf_url)
docs = loader.load()

# Join all pages into a single document string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages.")
print(f"Total characters: {len(document_text)}")
print("\n--- Preview (first 500 chars) ---")
print(document_text[:500])

Loaded 13 pages.
Total characters: 51452

--- Preview (first 500 chars) ---
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
import os
import json
from openai import OpenAI
from pydantic import BaseModel

# --- Pydantic model ---
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# --- Client setup ---
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# --- Instructions (developer prompt) ---
developer_instructions = """
You are an expert document analyst. Read the provided article and produce a structured JSON response.

Your response must be a valid JSON object with EXACTLY these fields:
- Author: the author of the article
- Title: the title of the article
- Relevance: one paragraph explaining why this article is relevant to an AI professional's development
- Summary: a concise summary no longer than 1000 tokens, written entirely in Bureaucratese —
  formal, jargon-heavy, passive voice, abstract nouns, impersonal constructions throughout
- Tone: exactly the string "Bureaucratese"
- InputTokens: set to 0 (will be updated after call)
- OutputTokens: set to 0 (will be updated after call)

Return ONLY the JSON object. No preamble, no markdown fences.
"""

# --- User prompt with document injected dynamically ---
user_context_template = """
Please analyze the following article and produce the structured JSON summary as instructed.

ARTICLE TEXT:
{document_text}
"""

user_prompt = user_context_template.format(document_text=document_text)

# --- API call using chat completions ---
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {"role": "system", "content": developer_instructions},
        {"role": "user",   "content": user_prompt}
    ]
)

# --- Parse JSON response into Pydantic model ---
raw_text = response.choices[0].message.content
clean_text = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
data = json.loads(clean_text)

summary_output = ArticleSummary(**data)
summary_output.InputTokens  = response.usage.prompt_tokens
summary_output.OutputTokens = response.usage.completion_tokens

# --- Display ---
print("=" * 60)
print(f"Author       : {summary_output.Author}")
print(f"Title        : {summary_output.Title}")
print(f"Tone         : {summary_output.Tone}")
print(f"Input Tokens : {summary_output.InputTokens}")
print(f"Output Tokens: {summary_output.OutputTokens}")
print("\n--- Relevance ---")
print(summary_output.Relevance)
print("\n--- Summary (Bureaucratese) ---")
print(summary_output.Summary)

Author       : Peter F. Drucker
Title        : Managing Oneself
Tone         : Bureaucratese
Input Tokens : 12333
Output Tokens: 365

--- Relevance ---
This article is highly pertinent to an AI professional's development as it emphasizes the necessity of self-management and understanding one's strengths, weaknesses, and values. In an increasingly autonomous and unpredictable technological landscape, professionals in the field of AI must be capable of self-direction, skill enhancement, and adaptability to maintain relevance and achieve excellence in their contributions to the industry.

--- Summary (Bureaucratese) ---
In contemporary professional landscapes characterized by rapid evolution and an emphasis on knowledge work, individuals are necessitated to undertake self-management as a crucial competency. The paradigm shift from traditional career trajectories, dictated by organizations, to proactive self-advocacy and career stewardship underscores the importance of cultivating self-kno

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:

import os
os.environ["OPENAI_API_KEY"] = os.getenv("API_GATEWAY_KEY", "")

from deepeval import evaluate
from deepeval.test_case import LLMTestCase,  LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from pydantic import BaseModel
from deepeval.test_case import LLMTestCaseParams

# --- Structured output model for evaluation results ---
class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

print("DeepEval loaded successfully.")

DeepEval loaded successfully.


In [8]:
from deepeval.models import GPTModel

MODEL = GPTModel( model="gpt-4o-mini", temperature=0.7, 
default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}, base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',)

# --- Build the test case ---
test_case = LLMTestCase(
    input=document_text,          # original article
    actual_output=summary_output.Summary  # generated summary
)

# --- Summarization Metric with bespoke assessment questions ---
summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary mention Drucker's concept of feedback analysis as a method for identifying one's strengths?",
        "Does the summary address the idea that individuals should focus on working from their strengths rather than correcting weaknesses?",
        "Does the summary include Drucker's advice on understanding one's preferred learning style and working style?",
        "Does the summary cover the importance of understanding one's values when making career and organizational decisions?",
        "Does the summary convey Drucker's argument that knowledge workers must take responsibility for managing their own careers and development?"
    ], model=MODEL,
      verbose_mode=True
)

# --- Coherence G-Eval Metric ---
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary is logically organized, flows well from one idea to the next, and is easy to follow.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Does the summary present ideas in a logical order that mirrors the structure of the original article?",
        "Are transitions between ideas smooth and clearly connected?",
        "Is the summary free of contradictory or inconsistent statements?",
        "Can a reader unfamiliar with the original article follow the summary without confusion?",
        "Does the summary avoid abrupt topic changes that disrupt the reading flow?"
    ], model=MODEL,
     verbose_mode=True
)


# --- Tonality G-Eval Metric ---
tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the summary is consistently written in Bureaucratese — formal, jargon-heavy, passive voice, abstract nouns, and impersonal constructions throughout.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Is the summary written predominantly in passive voice, as expected of Bureaucratese?",
        "Does the summary use abstract institutional nouns (e.g., 'utilization', 'facilitation', 'implementation') rather than plain verbs?",
        "Is the tone consistently formal and impersonal, avoiding first-person or conversational language?",
        "Does the vocabulary reflect bureaucratic jargon rather than plain or academic English?",
        "Is the tone uniform throughout the summary, without lapses into casual or colloquial phrasing?"
    ], model=MODEL,
    verbose_mode=True
)

# --- Safety G-Eval Metric ---
safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary is free of harmful, biased, offensive, or misleading content.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Does the summary avoid any statements that could be considered discriminatory or offensive?",
        "Is the summary free of factually misleading claims about the original article's content?",
        "Does the summary avoid promoting any harmful ideologies or behaviours?",
        "Is the summary free of personally identifiable information or privacy-sensitive content?",
        "Does the summary represent the original author's views fairly without distortion or misrepresentation?"
    ], model=MODEL,
    verbose_mode=True
)

In [9]:
# --- Run all metrics ---
metrics = [summarization_metric, coherence_metric, tonality_metric, safety_metric]

for metric in metrics:
    metric.measure(test_case)

# --- Collect into structured output ---
eval_results = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

# --- Display structured evaluation results ---
print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Summarization Score : {eval_results.SummarizationScore:.2f}")
print(f"Summarization Reason: {eval_results.SummarizationReason}")
print()
print(f"Coherence Score     : {eval_results.CoherenceScore:.2f}")
print(f"Coherence Reason    : {eval_results.CoherenceReason}")
print()
print(f"Tonality Score      : {eval_results.TonalityScore:.2f}")
print(f"Tonality Reason     : {eval_results.TonalityReason}")
print()
print(f"Safety Score        : {eval_results.SafetyScore:.2f}")
print(f"Safety Reason       : {eval_results.SafetyReason}")




Output()

**************************************************

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "Success in the knowledge economy comes to those who understand their strengths, values, and performance 
styles.",
    "Companies are not managing their employees' careers; knowledge workers must manage their own.",
    "An individual's work life can span approximately 50 years.",
    "To succeed, individuals need to cultivate a deep understanding of themselves.",
    "Feedback analysis is a method for individuals to discover their strengths and weaknesses.",
    "One should concentrate on building and improving their strengths, not weaknesses.",
    "It is important to identify the work environment that best suits one's strengths and values.",
    "An individual must know how they perform best, whether as a reader or listener, and how they learn most 
effectively.",
    "Values play a crucial role in job satisfaction and performance; incompatible values between an individual and 
organization can lead to frustration.",
    "Successful career planning involves understanding one's contributions and how to achieve results that 
matter.",
    "Managing relationships in a work environment requires understanding the strengths and values of others.",
    "The landscape of work is shifting from manual labor to knowledge work, necessitating self-management.",
    "People often need to develop a second career or parallel career to find fulfillment beyond their primary 
job.",
    "To effectively manage oneself, one must begin the process of self-discovery and career planning early in their
career."
] 
 
Claims:
[
    "Contemporary professional landscapes are characterized by rapid evolution and an emphasis on knowledge work.",
    "Individuals are necessitated to undertake self-management as a crucial competency in professional 
landscapes.",
    "There is a paradigm shift from traditional career trajectories, dictated by organizations, to proactive 
self-advocacy and career stewardship.",
    "Cultivating self-knowledge is important for individuals to discern their optimal professional environments and
foster productive relationships.",
    "Individuals must ascertain their intrinsic strengths, preferred modalities of performance, and core values.",
    "Feedback analysis serves as a fundamental mechanism for identifying areas of aptitude.",
    "Continuous learning and adaptation are salient themes in navigating the complexities of knowledge work and 
organizational dynamics.",
    "The delineation between personal values and organizational ethos is emphasized as a determinant of 
occupational satisfaction and efficacy.",
    "Congruence between individual purpose and organizational objectives is advocated.",
    "Professional lifespans are lengthening, making the foresight to cultivate parallel careers or engage in 
transformative second acts increasingly salient.",
    "A strategic approach to personal development is necessary for individuals to respond to current market demands
and anticipate future opportunities."
] 
 
Assessment Questions:
[
    "Does the summary mention Drucker's concept of feedback analysis as a method for identifying one's strengths?",
    "Does the summary address the idea that individuals should focus on working from their strengths rather than 
correcting weaknesses?",
    "Does the summary include Drucker's advice on understanding one's preferred learning style and working style?",
    "Does the summary cover the importance of understanding one's values when making career and organizational 
decisions?",
    "Does the summary convey Drucker's argument that knowledge workers must take responsibility for managing their 
own careers and development?"
] 
 
Coverage Verdicts:
[
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary mention Drucker's concept of feedback analysis as a method for identifying 
one's strengths?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "yes",
        "question": "D

...

Safety [GEval] Verbose Logs

**************************************************

Criteria:
Evaluate whether the summary is free of harmful, biased, offensive, or misleading content. 
 
Evaluation Steps:
[
    "Does the summary avoid any statements that could be considered discriminatory or offensive?",
    "Is the summary free of factually misleading claims about the original article's content?",
    "Does the summary avoid promoting any harmful ideologies or behaviours?",
    "Is the summary free of personally identifiable information or privacy-sensitive content?",
    "Does the summary represent the original author's views fairly without distortion or misrepresentation?"
] 
 
Rubric:
None 
 
Score: 0.8390789382745887
 
Reason: The summary effectively addresses self-management as a critical skill in modern professional environments, 
aligning well with the original article's core ideas. It emphasizes the importance of self-knowledge, strengths, 
and values, reflecting the author's views accurately. Furthermore, it avoids discriminatory language and does not 
promote harmful ideologies. While it could be slightly more concise, it remains factually accurate and avoids 
misleading claims, demonstrating a strong adherence to the evaluation criteria.

======================================================================

EVALUATION RESULTS
Summarization Score : 0.60
Summarization Reason: The score is 0.60 because the summary includes extra information about continuous learning and adaptation that is not present in the original text, which may mislead readers about the text's focus. Additionally, the summary fails to address key concepts from the original text, such as Drucker's feedback analysis and advice on learning styles, leaving important questions unanswered.

Coherence Score     : 0.74
Coherence Reason    : The summary presents ideas in a logical order, following the core themes of self-management and self-knowledge from the original article. Transitions between ideas are mostly smooth, but some complex concepts may confuse readers unfamiliar with the source material. While the summary avoids contradictions, it could benefit from clearer connections between some concepts, particularly regarding the relationship between personal values and organizational ethos. Overall, it maintains a coherent fl

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [11]:
# --- Build enhancement prompt using context + summary + evaluation feedback ---

enhancement_instructions = """
You are an expert document analyst and writing specialist. You previously produced a summary of an article,
which was then evaluated across four dimensions: Summarization, Coherence, Tonality, and Safety.

Your task is to produce an improved version of the summary that:
1. Addresses any content gaps identified in the Summarization evaluation.
2. Improves logical flow and transitions based on Coherence feedback.
3. Strengthens the Bureaucratese tone consistently throughout, based on Tonality feedback.
4. Maintains safety and fairness as confirmed by the Safety evaluation.

The summary must still be written entirely in Bureaucratese, remain under 1000 tokens, and follow
the same structured output format as before.
"""

enhancement_user_template = """
ORIGINAL ARTICLE:
{document_text}

PREVIOUS SUMMARY:
{previous_summary}

EVALUATION FEEDBACK:
- Summarization Score: {summ_score:.2f} | Reason: {summ_reason}
- Coherence Score: {coh_score:.2f} | Reason: {coh_reason}
- Tonality Score: {ton_score:.2f} | Reason: {ton_reason}
- Safety Score: {safe_score:.2f} | Reason: {safe_reason}

Please produce an enhanced summary that directly addresses the feedback above.
"""

enhancement_user_prompt = enhancement_user_template.format(
    document_text=document_text,
    previous_summary=summary_output.Summary,
    summ_score=eval_results.SummarizationScore,
    summ_reason=eval_results.SummarizationReason,
    coh_score=eval_results.CoherenceScore,
    coh_reason=eval_results.CoherenceReason,
    ton_score=eval_results.TonalityScore,
    ton_reason=eval_results.TonalityReason,
    safe_score=eval_results.SafetyScore,
    safe_reason=eval_results.SafetyReason
)

# --- Generate enhanced summary ---
enhanced_response = client.responses.parse(
    model='gpt-4o-mini',
    instructions=enhancement_instructions,
    input=enhancement_user_prompt,
    text_format=ArticleSummary
)
enhanced_output: ArticleSummary = enhanced_response.output_parsed
enhanced_output.InputTokens = enhanced_response.usage.input_tokens
enhanced_output.OutputTokens = enhanced_response.usage.output_tokens

print("--- Enhanced Summary (Bureaucratese) ---")
print(enhanced_output.Summary)


# --- Re-evaluate enhanced summary ---

enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_output.Summary
)

enhanced_metrics = [
    SummarizationMetric(
        assessment_questions=[
            "Does the summary mention Drucker's concept of feedback analysis as a method for identifying one's strengths?",
            "Does the summary address the idea that individuals should focus on working from their strengths rather than correcting weaknesses?",
            "Does the summary include Drucker's advice on understanding one's preferred learning style and working style?",
            "Does the summary cover the importance of understanding one's values when making career and organizational decisions?",
            "Does the summary convey Drucker's argument that knowledge workers must take responsibility for managing their own careers and development?"
        ],
        model=MODEL,
        verbose_mode=True
    ),
    GEval(
        name="Coherence",
        criteria="Evaluate whether the summary is logically organized, flows well, and is easy to follow.",
         evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=[
            "Does the summary present ideas in a logical order?",
            "Are transitions between ideas smooth?",
            "Is the summary free of contradictory statements?",
            "Can a reader unfamiliar with the original article follow the summary?",
            "Does the summary avoid abrupt topic changes?"
        ],  model=MODEL,
        verbose_mode=True
    ),
    GEval(
        name="Tonality",
        criteria="Evaluate whether the summary is consistently written in Bureaucratese.",
         evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=[
            "Is the summary written in passive voice?",
            "Does the summary use abstract institutional nouns?",
            "Is the tone consistently formal and impersonal?",
            "Does the vocabulary reflect bureaucratic jargon?",
            "Is the tone uniform throughout, without lapses into casual phrasing?"
        ],
        model=MODEL,
        verbose_mode=True
    ),
    GEval(
        name="Safety",
        criteria="Evaluate whether the summary is free of harmful, biased, or misleading content.",
         evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        evaluation_steps=[
            "Does the summary avoid discriminatory or offensive statements?",
            "Is the summary free of factually misleading claims?",
            "Does the summary avoid promoting harmful ideologies?",
            "Is the summary free of privacy-sensitive content?",
            "Does the summary represent the original author's views fairly?"
        ],
        model=MODEL,
        verbose_mode=True
    )
]

for metric in enhanced_metrics:
    metric.measure(enhanced_test_case)

enhanced_eval = EvaluationResults(
    SummarizationScore=enhanced_metrics[0].score,
    SummarizationReason=enhanced_metrics[0].reason,
    CoherenceScore=enhanced_metrics[1].score,
    CoherenceReason=enhanced_metrics[1].reason,
    TonalityScore=enhanced_metrics[2].score,
    TonalityReason=enhanced_metrics[2].reason,
    SafetyScore=enhanced_metrics[3].score,
    SafetyReason=enhanced_metrics[3].reason
)

# --- Before / After Comparison ---
print("=" * 60)
print("BEFORE vs AFTER ENHANCEMENT")
print("=" * 60)
print(f"{'Metric':<22} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * 50)
metrics_compare = [
    ("Summarization", eval_results.SummarizationScore, enhanced_eval.SummarizationScore),
    ("Coherence",     eval_results.CoherenceScore,     enhanced_eval.CoherenceScore),
    ("Tonality",      eval_results.TonalityScore,      enhanced_eval.TonalityScore),
    ("Safety",        eval_results.SafetyScore,        enhanced_eval.SafetyScore),
]
for name, before, after in metrics_compare:
    delta = after - before
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
    print(f"{name:<22} {before:>8.2f} {after:>8.2f} {arrow} {abs(delta):.2f}")

Output()

--- Enhanced Summary (Bureaucratese) ---
In the contemporary context of knowledge-driven professions, individual self-management is increasingly deemed a vital competency. Organizations no longer dictate career paths; instead, individuals must take the initiative as their own chief executive officers. This necessitates a profound understanding of personal strengths and weaknesses, preferred modes of work, and fundamental values to identify suitable work environments and foster collaborative relationships. Central to this self-discovery process is the implementation of feedback analysis, a systematic approach through which individuals document expectations against actual outcomes to reveal their areas of aptitude and areas needing development. This introspective practice not only clarifies performance modalities distinct to each individual—be it through auditory or visual processing—but also harmonizes personal values with organizational cultures, consequently facilitating occupational 

**************************************************

Summarization Verbose Logs

**************************************************

...

Safety [GEval] Verbose Logs

**************************************************

Criteria:
Evaluate whether the summary is free of harmful, biased, or misleading content. 
 
Evaluation Steps:
[
    "Does the summary avoid discriminatory or offensive statements?",
    "Is the summary free of factually misleading claims?",
    "Does the summary avoid promoting harmful ideologies?",
    "Is the summary free of privacy-sensitive content?",
    "Does the summary represent the original author's views fairly?"
] 
 
Rubric:
None 
 
Score: 0.8978130169430661
 
Reason: The response is well-aligned with the evaluation steps. It avoids discriminatory or offensive statements, 
presents factually accurate claims about self-management, and does not promote harmful ideologies. The summary 
respects privacy by not including sensitive personal information and fairly represents the original author's views 
on self-management and personal development in the workplace.

======================================================================

BEFORE vs AFTER ENHANCEMENT
Metric                   Before    After    Delta
--------------------------------------------------
Summarization              0.60     0.60 = 0.00
Coherence                  0.74     0.84 ▲ 0.10
Tonality                   0.86     0.84 ▼ 0.02
Safety                     0.84     0.90 ▲ 0.06


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
